In [ ]:
import pandas as pd
import os
import pandas as pd
import os
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

import cv2

# **Visual Data**

In [ ]:
#List all Pickle (.pkl) files

pklfiles = [f for f in os.listdir('Project_Features') if os.path.isfile(os.path.join('Project_Features', f)) and f.endswith('_visual.pkl')]
print(pklfiles)

In [ ]:
# Load one
for video in pklfiles:
    print(video)
    data = pd.read_pickle(os.path.join('Project_Features', video))

# Print first lines
print(data.head())

In [ ]:
#Access one frame
data.iloc[0]

In [ ]:
#Lets visualize some features in the images - Poses
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

frame = 1960

frame_id = data.iloc[frame]['Frame']
print(frame_id)

img = Image.open(frame_id)

fig = plt.figure()
ax = fig.add_axes([0, 0, 1, 1])
ax.imshow(img)

for person in data.iloc[frame]['Poses']:

    box = person['bbox']
    # Create a rectangle patch
    rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], linewidth=3, edgecolor='w', facecolor='none')

    # Add patch to the image
    ax.add_patch(rect)

    for kp in person['pose']:

        if kp[2] > 0.5:
            kx, ky = int(kp[0]), int(kp[1])
            # Only draw if the point isn't 0,0 (undetected)
            keypoint = patches.Circle((kx, ky), 10)
            ax.add_patch(keypoint)



In [ ]:
# Facial emotion recognition

fers = data.iloc[frame]['Fer']

print(fers[0].keys())



In [ ]:
print(fers[0]['bbox'])
print(fers[0]['top_emotion'])

print(fers[0]['probabilities'])

print(fers[0]['landmarks'])

In [ ]:
#Visualize emotions and detected faces

fig = plt.figure()
ax = fig.add_axes([0, 0, 1, 1])
ax.imshow(img)

fers = data.iloc[frame]['Fer']

for fer in fers:

    x1 = fer['bbox'][0]
    y1 = fer['bbox'][1]
    x2 = fer['bbox'][2]
    y2 = fer['bbox'][3]

    print(fer['top_emotion'])

    # Create a rectangle patch
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=4, edgecolor='g', facecolor='none')

    # Add patch to the image
    ax.add_patch(rect)

    ax.text(x1, y1, fer['top_emotion'],
            verticalalignment='top',
            color='w')
    
    # We can also visualize the landmarks
    landmarks = fer['landmarks']
    for kp in landmarks:
        
        kx, ky = int(kp[0]), int(kp[1])

        keypoint = patches.Circle((kx, ky), 5)
        ax.add_patch(keypoint)

plt.show()

In [ ]:
### We can also look at the OCR output


pklfiles = [f for f in os.listdir('Project_Features') if os.path.isfile(os.path.join('Project_Features', f)) and f.endswith('_ocr.pkl')]

# Load one
for video in pklfiles:
    print(video)
    data = pd.read_pickle(os.path.join('Project_Features', video))

# Print first lines
print(data.head())

frame = 3000

frame_id = data.iloc[frame]['Frame']
print(frame_id)

img = Image.open(frame_id)

fig = plt.figure()
ax = fig.add_axes([0, 0, 1, 1])
ax.imshow(img)

for t in data.iloc[frame]['OCR']:

    box = t['bbox']
    # Create a rectangle patch
    rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], linewidth=10, edgecolor='w', facecolor='none')

    # Add patch to the image
    ax.add_patch(rect)

    print(t['text'])

# **Audio & Speech**

In [ ]:
#List all Pickle (.pkl) files

pklfiles = [f for f in os.listdir('Project_Features') if os.path.isfile(os.path.join('Project_Features', f)) and f.endswith('_audio.pkl')]
print(pklfiles)

In [ ]:
# Load one
video = 'Ventura_vs_Marques_Mendes_November_25'
data_audio = pd.read_pickle(os.path.join('Project_Features', video + '_audio.pkl'))

# Print first lines
data_audio.head()

In [ ]:
print(data_audio.keys())

In [ ]:
print(data_audio['time stamp'][19:25])
print(data_audio['duration'][19:25])

In [ ]:
print(data_audio.iloc[13])

In [ ]:
#Lets listen
from scipy.io import wavfile
from IPython.display import Audio

samplerate, audio = wavfile.read(os.path.join('Audio', video + '.wav'))

sample = audio[int(data_audio.iloc[13]['time stamp']*samplerate):int(data_audio.iloc[13]['time stamp']+data_audio.iloc[13]['duration'])*samplerate,0]

display(Audio(sample,rate=samplerate))

In [ ]:
#List all Pickle (.pkl) files

pklfiles = [f for f in os.listdir('Project_Features') if os.path.isfile(os.path.join('Project_Features', f)) and f.endswith('_speech.pkl')]
print(pklfiles)

In [ ]:
# Load one
video = 'Ventura_vs_Marques_Mendes_November_25_speech'
data_speech = pd.read_pickle(os.path.join('Project_Features', video + '.pkl'))

# Print first lines
data_speech.head()

In [ ]:
#Verify the transcritp
transcript = data_speech.iloc[13]['transcript']
print(transcript)

# **Text**

In [ ]:
# Bag-of-Words

from glob import glob

def get_files():
    files = glob('Project_Features/*_speech.pkl')
    return files

corpus = []

trans = get_files()

print(trans)

for file in trans:
    df = pd.read_pickle(file)
    texts = df['transcript']
    for text in texts:
        corpus.append(text)

In [ ]:
#For a single video
from collections import Counter
def create_bow(text):
  """Creates a Bag-of-Words representation from text.

  Args:
    text: The input text string.

  Returns:
    A dictionary where keys are words and values are their frequencies.
  """
  words = text.lower().split()  # Convert to lowercase and split into words
  word_counts = Counter(words)  # Count word frequencies
  return dict(word_counts)  # Return as a dictionary

bow = create_bow(''.join(corpus))
print(len(bow.keys()))

print(bow)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(vocabulary = bow.keys())


In [ ]:
texts = data_speech['transcript']

count_matrix = vectorizer.transform(texts)
BoW = vectorizer.get_feature_names_out()
count_array = count_matrix.toarray()

print(count_array.shape)

for i in range(count_array.shape[0]):
  words = np.where(np.squeeze(count_array[i,:]) != 0)

  print(BoW[words])

In [ ]:
#Load the single file for pool timeline
data_polls = pd.read_pickle(os.path.join('Project_Features', 'polls_timeline.pkl'))

# Print first lines
data_polls.head()

In [ ]:
#Load the single file for parties
data_parties = pd.read_pickle(os.path.join('Project_Features', 'candidate_party.pkl'))

# Print first lines
data_parties.head()